In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, f1_score, top_k_accuracy_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

TEXT, LABEL = "input_text", "output_text"

In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(train.shape, test.shape)
train.head()

In [ ]:
norm = lambda s: " ".join(str(s).lower().split())
train_texts = set(train[TEXT].map(norm))
dupes = test[TEXT].map(norm).isin(train_texts).sum()
print(f"Exact duplicates between test and train: {dupes} of {len(test)}")

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english",
                              ngram_range=(1, 2), sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=2000)),
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C": [0.5, 1, 5, 10],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(pipeline, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
search.fit(train[TEXT], train[LABEL])

print("Best settings:", search.best_params_)
print(f"Cross-validation macro F1: {search.best_score_:.3f}")

In [ ]:
model = search.best_estimator_
pred = model.predict(test[TEXT])
proba = model.predict_proba(test[TEXT])

acc = accuracy_score(test[LABEL], pred)
macro_f1 = f1_score(test[LABEL], pred, average="macro")
top3 = top_k_accuracy_score(test[LABEL], proba, k=3, labels=model.classes_)

print(f"TEST accuracy:       {acc:.3f}")
print(f"TEST macro F1:       {macro_f1:.3f}")
print(f"TEST top-3 accuracy: {top3:.3f}")
print(classification_report(test[LABEL], pred, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay.from_predictions(test[LABEL], pred, ax=ax,
                                        xticks_rotation=90, colorbar=False)
ax.set_title("Baseline (TF-IDF + logistic regression): test set")
plt.tight_layout()
plt.savefig("results/baseline_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
errors = test[pred != test[LABEL]].assign(predicted=pred[pred != test[LABEL]])
print(f"Mistakes: {len(errors)} of {len(test)}")
errors[[LABEL, "predicted", TEXT]]

In [ ]:
with open("results/baseline_results.json", "w") as f:
    json.dump({"model": "tfidf + logistic regression",
               "best_params": {k: str(v) for k, v in search.best_params_.items()},
               "cv_macro_f1": round(search.best_score_, 4),
               "test_accuracy": round(acc, 4),
               "test_macro_f1": round(macro_f1, 4),
               "test_top3_accuracy": round(top3, 4),
               "test_train_exact_duplicates": int(dupes)}, f, indent=2)
print("Saved results/baseline_results.json")